# BCG GenAI — AI-Powered Financial Chatbot

**Developer:** Mike (Junior Data Scientist, GenAI Consulting Team)

**Client:** Global Finance Corp. (GFC)

**Date:** April 2026

---

## Overview

This notebook implements an AI-powered financial chatbot that analyses financial data from 10-K filings and provides interactive insights. The chatbot uses:

- **Rule-based logic** for handling predefined financial queries
- **State management** to remember conversation context and previous queries
- **Error handling** to gracefully manage unrecognised inputs
- **Interactive dialogue** to guide users toward relevant insights

The chatbot integrates the financial data extracted and analysed in Task 1 (Microsoft, Tesla, Apple — FY2022-2024).

---
## Step 1: Load and Structure Financial Data

In [ ]:
import pandas as pd

# Load the cleaned financial data from Task 1
df = pd.read_csv('ai_chatbot_data.csv')

print('Financial data loaded successfully.')
print(f'Companies: {df["Company"].unique().tolist()}')
print(f'Fiscal Years: {sorted(df["Fiscal Year"].unique().tolist())}')
print(f'Metrics available: {[c for c in df.columns if c not in ["Company", "Fiscal Year"]]}')
print()
df

---
## Step 2: Build the Financial Chatbot

The chatbot is built around three core components:

1. **Data retrieval layer** — maps user queries to specific data lookups
2. **Conversation engine** — handles dialogue flow, state, and error handling
3. **Insight generator** — produces human-readable financial analysis from raw data

In [ ]:
class FinancialChatbot:
    """AI-powered financial chatbot for analysing 10-K filing data.
    
    Uses rule-based logic with state management to provide
    interactive financial insights across multiple companies.
    """
    
    def __init__(self, data: pd.DataFrame):
        """Initialise chatbot with financial data."""
        self.data = data
        self.companies = [c.lower() for c in data['Company'].unique()]
        self.company_names = {c.lower(): c for c in data['Company'].unique()}
        self.years = sorted(data['Fiscal Year'].unique())
        
        # State management
        self.context = {
            'last_company': None,
            'last_metric': None,
            'last_year': None,
            'query_count': 0,
            'history': [],
        }
        
        # Map keywords to data columns
        self.metric_map = {
            'revenue': 'Total Revenue',
            'sales': 'Total Revenue',
            'income': 'Net Income',
            'profit': 'Net Income',
            'earnings': 'Net Income',
            'net income': 'Net Income',
            'assets': 'Total Assets',
            'liabilities': 'Total Liabilities',
            'debt': 'Total Liabilities',
            'cash flow': 'Cash Flow from Operations',
            'cash': 'Cash Flow from Operations',
            'operating cash': 'Cash Flow from Operations',
            'margin': 'Profit Margin (%)',
            'profit margin': 'Profit Margin (%)',
            'equity': 'Shareholders Equity',
            'growth': 'Total Revenue Growth (%)',
            'revenue growth': 'Total Revenue Growth (%)',
            'income growth': 'Net Income Growth (%)',
        }
    
    def _format_currency(self, value):
        """Format large numbers into readable currency strings."""
        if abs(value) >= 1e12:
            return f'${value/1e12:.2f} trillion'
        elif abs(value) >= 1e9:
            return f'${value/1e9:.1f} billion'
        elif abs(value) >= 1e6:
            return f'${value/1e6:.1f} million'
        else:
            return f'${value:,.0f}'
    
    def _detect_company(self, query):
        """Detect which company the user is asking about."""
        query_lower = query.lower()
        for key, name in self.company_names.items():
            if key in query_lower:
                return name
        # Check for common abbreviations
        abbrevs = {'msft': 'Microsoft', 'aapl': 'Apple', 'tsla': 'Tesla'}
        for abbrev, name in abbrevs.items():
            if abbrev in query_lower:
                return name
        # Fall back to context
        return self.context['last_company']
    
    def _detect_metric(self, query):
        """Detect which financial metric the user is asking about."""
        query_lower = query.lower()
        # Check longer phrases first to avoid partial matches
        for keyword in sorted(self.metric_map.keys(), key=len, reverse=True):
            if keyword in query_lower:
                return self.metric_map[keyword]
        return self.context['last_metric']
    
    def _detect_year(self, query):
        """Detect which fiscal year the user is asking about."""
        for year in self.years:
            if str(year) in query:
                return year
        return None  # None means all years
    
    def _get_metric_data(self, company, metric, year=None):
        """Retrieve specific financial data."""
        company_data = self.data[self.data['Company'] == company]
        if year:
            row = company_data[company_data['Fiscal Year'] == year]
            if row.empty:
                return None
            return row[metric].values[0]
        return company_data[['Fiscal Year', metric]]
    
    def _generate_trend_insight(self, company, metric):
        """Generate a trend analysis insight for a metric."""
        company_data = self.data[self.data['Company'] == company].sort_values('Fiscal Year')
        values = company_data[metric].values
        years = company_data['Fiscal Year'].values
        
        if len(values) < 2:
            return 'Insufficient data for trend analysis.'
        
        # Calculate overall change
        overall_change = ((values[-1] - values[0]) / abs(values[0])) * 100
        latest_change = ((values[-1] - values[-2]) / abs(values[-2])) * 100
        
        # Determine trend direction
        if all(values[i] <= values[i+1] for i in range(len(values)-1)):
            trend = 'consistently increasing'
        elif all(values[i] >= values[i+1] for i in range(len(values)-1)):
            trend = 'consistently declining'
        else:
            trend = 'mixed (fluctuating)'
        
        # Build response based on whether metric is a percentage or currency
        is_percentage = '%' in metric
        if is_percentage:
            values_str = ', '.join([f'FY{y}: {v:.1f}%' for y, v in zip(years, values)])
        else:
            values_str = ', '.join([f'FY{y}: {self._format_currency(v)}' for y, v in zip(years, values)])
        
        insight = f"""\n{metric} for {company} (FY{years[0]}-{years[-1]}):
  {values_str}
  Trend: {trend}
  Overall change: {overall_change:+.1f}%
  Latest YoY change: {latest_change:+.1f}%"""
        
        return insight
    
    def _handle_comparison(self, metric, year=None):
        """Compare a metric across all companies."""
        if not year:
            year = max(self.years)
        
        is_percentage = '%' in metric
        results = []
        for company in self.data['Company'].unique():
            value = self._get_metric_data(company, metric, year)
            if value is not None:
                if is_percentage:
                    results.append((company, value, f'{value:.1f}%'))
                else:
                    results.append((company, value, self._format_currency(value)))
        
        # Sort by value descending
        results.sort(key=lambda x: x[1], reverse=True)
        
        response = f'\n{metric} comparison for FY{year}:\n'
        for i, (company, value, formatted) in enumerate(results, 1):
            response += f'  {i}. {company}: {formatted}\n'
        
        leader = results[0][0]
        response += f'\n  Leader: {leader}'
        
        return response
    
    def _handle_summary(self, company):
        """Generate a comprehensive financial summary for a company."""
        latest_year = max(self.years)
        row = self.data[(self.data['Company'] == company) & (self.data['Fiscal Year'] == latest_year)].iloc[0]
        
        response = f"""\nFinancial Summary for {company} (FY{latest_year}):
  Revenue:        {self._format_currency(row['Total Revenue'])}
  Net Income:     {self._format_currency(row['Net Income'])}
  Profit Margin:  {row['Profit Margin (%)']:.1f}%
  Total Assets:   {self._format_currency(row['Total Assets'])}
  Total Liabilities: {self._format_currency(row['Total Liabilities'])}
  Shareholders Equity: {self._format_currency(row['Shareholders Equity'])}
  Operating Cash Flow: {self._format_currency(row['Cash Flow from Operations'])}
  Debt-to-Asset Ratio: {row['Debt to Asset Ratio']:.3f}"""
        
        # Add revenue growth if available
        if pd.notna(row.get('Total Revenue Growth (%)', None)):
            response += f"\n  Revenue Growth (YoY): {row['Total Revenue Growth (%)']:+.1f}%"
        if pd.notna(row.get('Net Income Growth (%)', None)):
            response += f"\n  Net Income Growth (YoY): {row['Net Income Growth (%)']:+.1f}%"
        
        # Add a brief health assessment
        margin = row['Profit Margin (%)']
        dar = row['Debt to Asset Ratio']
        
        if margin > 20 and dar < 0.6:
            assessment = 'Strong — high profitability with manageable leverage.'
        elif margin > 10 and dar < 0.7:
            assessment = 'Healthy — solid profitability, leverage within acceptable range.'
        elif margin > 5:
            assessment = 'Moderate — profitable but may face margin or leverage pressure.'
        else:
            assessment = 'Concerning — low margins may indicate operational challenges.'
        
        response += f'\n\n  Overall Assessment: {assessment}'
        
        return response
    
    def _suggest_follow_ups(self, company=None, metric=None):
        """Suggest related queries to encourage exploration."""
        suggestions = []
        
        if company and not metric:
            suggestions = [
                f'Try: "What is {company}\'s revenue trend?"',
                f'Try: "Show me {company}\'s profit margin"',
                f'Try: "Compare {company} to others"',
            ]
        elif metric and not company:
            suggestions = [
                'Try specifying a company: "Microsoft", "Tesla", or "Apple"',
                f'Try: "Compare {metric} across companies"',
            ]
        elif company and metric:
            other_metrics = ['revenue', 'profit margin', 'cash flow', 'assets']
            suggestions = [
                f'Try: "Compare {metric} across all companies"',
                f'Try: "Give me a full summary of {company}"',
            ]
        else:
            suggestions = [
                'Try: "Show me Apple\'s revenue"',
                'Try: "Compare profit margins"',
                'Try: "Give me a summary of Tesla"',
                'Try: "Which company has the highest revenue?"',
            ]
        
        return '\n  '.join(suggestions)
    
    def respond(self, user_input: str) -> str:
        """Process user input and generate a response."""
        self.context['query_count'] += 1
        self.context['history'].append(user_input)
        query = user_input.lower().strip()
        
        # Handle greetings
        if query in ['hi', 'hello', 'hey', 'help', 'start']:
            return """Hello! I'm your Financial Analysis Assistant.
I can help you explore financial data for Microsoft, Tesla, and Apple (FY2022-2024).

Here's what I can do:
  - Look up specific metrics (revenue, income, assets, liabilities, cash flow)
  - Show trends over time for any company
  - Compare metrics across companies
  - Provide full financial summaries

What would you like to know?"""
        
        # Handle exit
        if query in ['quit', 'exit', 'bye', 'goodbye']:
            return f'Thanks for using the Financial Assistant! You asked {self.context["query_count"]} questions. Goodbye!'
        
        # Detect intent
        company = self._detect_company(query)
        metric = self._detect_metric(query)
        year = self._detect_year(query)
        is_comparison = any(word in query for word in ['compare', 'comparison', 'vs', 'versus', 'highest', 'lowest', 'best', 'worst', 'rank'])
        is_summary = any(word in query for word in ['summary', 'overview', 'overall', 'health', 'snapshot'])
        is_trend = any(word in query for word in ['trend', 'over time', 'history', 'change', 'grew', 'declined'])
        
        # Update context
        if company:
            self.context['last_company'] = company
        if metric:
            self.context['last_metric'] = metric
        if year:
            self.context['last_year'] = year
        
        # Route to appropriate handler
        try:
            # Summary request
            if is_summary and company:
                return self._handle_summary(company)
            
            # Comparison request
            if is_comparison and metric:
                return self._handle_comparison(metric, year)
            
            # Trend request
            if is_trend and company and metric:
                return self._generate_trend_insight(company, metric)
            
            # Specific data point
            if company and metric and year:
                value = self._get_metric_data(company, metric, year)
                if value is not None:
                    is_pct = '%' in metric
                    formatted = f'{value:.1f}%' if is_pct else self._format_currency(value)
                    response = f'{company}\'s {metric} in FY{year}: {formatted}'
                    response += f'\n\n  Want to explore more?\n  {self._suggest_follow_ups(company, metric)}'
                    return response
            
            # Company + metric (no year) → show all years
            if company and metric:
                return self._generate_trend_insight(company, metric)
            
            # Just company → suggest metrics
            if company and not metric:
                return f"""I can look up data for {company}. What metric are you interested in?

  Available metrics: revenue, net income, profit margin, assets,
  liabilities, cash flow, equity, growth rates

  {self._suggest_follow_ups(company)}"""
            
            # Just metric → ask for company or compare
            if metric and not company:
                return f"""Which company would you like to see {metric} for?

  Available: Microsoft, Tesla, Apple
  Or I can compare all three — just say \"compare\".

  {self._suggest_follow_ups(None, metric)}"""
            
            # Nothing detected → error handling
            return f"""I'm not sure I understood that. I can help with financial data for Microsoft, Tesla, and Apple.

  {self._suggest_follow_ups()}

  Type \"help\" for a full list of what I can do."""
        
        except Exception as e:
            return f'Sorry, I encountered an error processing your request: {str(e)}\nPlease try rephrasing your question.'

print('FinancialChatbot class defined successfully.')

---
## Step 3: Initialise and Test the Chatbot

Let's put the chatbot through a series of tests to verify it handles all query types correctly.

In [ ]:
# Initialise the chatbot with our financial data
bot = FinancialChatbot(df)
print('Chatbot initialised with data for:', df['Company'].unique().tolist())
print('Fiscal years covered:', sorted(df['Fiscal Year'].unique().tolist()))

### Test 1: Greeting and Help

In [ ]:
print(bot.respond('hello'))

### Test 2: Specific Data Point Query

In [ ]:
print(bot.respond('What was Apple\'s revenue in 2024?'))

In [ ]:
print(bot.respond('What is Microsoft\'s net income in 2023?'))

### Test 3: Trend Analysis

In [ ]:
print(bot.respond('Show me Tesla\'s revenue trend'))

In [ ]:
print(bot.respond('How has Apple\'s profit margin changed over time?'))

### Test 4: Cross-Company Comparison

In [ ]:
print(bot.respond('Compare revenue across all companies'))

In [ ]:
print(bot.respond('Which company has the highest profit margin?'))

### Test 5: Full Company Summary

In [ ]:
print(bot.respond('Give me a financial summary of Microsoft'))

In [ ]:
print(bot.respond('What is Tesla\'s overall financial health?'))

### Test 6: State Management (Context Awareness)

In [ ]:
# First ask about Apple
print('Query 1:')
print(bot.respond('Tell me about Apple\'s revenue'))
print()

# Then ask a follow-up without specifying Apple — chatbot should remember context
print('Query 2 (follow-up, no company specified):')
print(bot.respond('What about the cash flow?'))

### Test 7: Error Handling

In [ ]:
# Unrecognised query
print(bot.respond('What is the weather like today?'))

In [ ]:
# Company without metric
print(bot.respond('Tell me about Tesla'))

### Test 8: Interactive Conversation Demo

In [ ]:
# Simulate a full conversation
demo_bot = FinancialChatbot(df)

conversation = [
    'hi',
    'What was Microsoft\'s revenue in 2024?',
    'How does that compare to the other companies?',
    'Show me Tesla\'s profit margin trend',
    'Give me a full summary of Apple',
    'bye',
]

print('=' * 70)
print('INTERACTIVE CONVERSATION DEMO')
print('=' * 70)

for query in conversation:
    print(f'\nUser: {query}')
    print(f'\nBot: {demo_bot.respond(query)}')
    print('-' * 50)

---
## Step 4: Live Interactive Mode

Uncomment and run the cell below to chat with the bot interactively in Jupyter.

In [ ]:
# Uncomment to run interactive mode:

# live_bot = FinancialChatbot(df)
# print(live_bot.respond('hello'))
# print()
# while True:
#     user_input = input('You: ')
#     if user_input.lower() in ['quit', 'exit', 'bye']:
#         print(f'Bot: {live_bot.respond(user_input)}')
#         break
#     print(f'Bot: {live_bot.respond(user_input)}\n')

---
## Findings and Design Decisions

### Architecture

The chatbot uses a **rule-based architecture** with four layers:

1. **Intent detection** — keyword matching to identify what the user wants (data point, trend, comparison, summary)
2. **Entity extraction** — detecting company names, metrics, and fiscal years from natural language
3. **Data retrieval** — fetching the right data from the structured DataFrame
4. **Response generation** — formatting raw data into readable, insightful responses

### Key Design Decisions

- **State management:** The chatbot remembers the last company and metric discussed, enabling natural follow-up questions ("What about cash flow?" after asking about revenue)
- **Error handling:** Unrecognised queries return helpful suggestions rather than error messages, guiding users toward valid queries
- **Metric aliasing:** Multiple terms map to the same data ("profit", "earnings", "income" all map to Net Income) for a more natural user experience
- **Automated assessment:** The summary feature includes a financial health assessment based on profit margin and debt-to-asset ratio thresholds

### Capabilities Demonstrated

| Capability | Example Query | Response Type |
|---|---|---|
| Specific lookup | "Apple's revenue in 2024" | Single formatted value |
| Trend analysis | "Tesla's revenue trend" | Multi-year values + direction + % change |
| Cross-company comparison | "Compare profit margins" | Ranked list with leader highlighted |
| Financial summary | "Summary of Microsoft" | Full snapshot with health assessment |
| Context awareness | "What about cash flow?" | Uses last-mentioned company |
| Error recovery | "What's the weather?" | Helpful suggestions |

### Future Enhancements

For a production deployment, the chatbot could be enhanced with:
- **NLP integration** (e.g., spaCy or NLTK) for more robust entity extraction
- **Chart generation** — render matplotlib charts inline in response to trend queries
- **Larger dataset** — support for more companies and longer time horizons
- **LLM integration** — use a language model for more natural conversational responses
- **Web interface** — deploy via Flask or Streamlit for GFC's client-facing portal